#
# <center>**Machine Translation**</center>

Intent-based chatbots are limited by design. An alternative is building a sequence-to-sequence model. Here, translating a sentence in English to German.

Using PyTorch, the model implemented is the encoder-decoder, where the encoder and decoder have RNN layers, specifically LSTM, to predict individual words following the previous word. This model is a precursor to a more complex one.

Hence, a small dataset will be used. See why a limited dataset size severly restricts the model's performance.

In [ ]:
!pip install datasets evaluate --upgrade
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

#
### <center> **Data preprocessing** </center>

- Load data and split into training, test, validation sets
- Load tokenizers and split the texts (in English and German) into single tokens

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import spacy

import random
import pandas as pd
import datasets
from datasets import Dataset

In [ ]:
# Entirely optional to run this cell: set seed for reproducible results
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
# Load in the dataset and convert to a Dataset object
data = pd.read_csv("data.tsv", sep="\t")
dataset = Dataset.from_pandas(data)

######################################### Split the dataset into 80% training, 10% validation, 10% test #############################################
# First, split into 90% | 10%
dataset = dataset.train_test_split(test_size=0.10)
test_dataset = dataset["test"]
train_val = dataset["train"]

# Second split: from the 90%, validation - 10% and train - 80%
train_val_split = train_val.train_test_split(test_size=0.1111)

train_dataset = train_val_split["train"]
validation_dataset = train_val_split["test"]

dataset = {
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
}

# show dataset structure
dataset

{'train': Dataset({
     features: ['de', 'en'],
     num_rows: 51
 }),
 'validation': Dataset({
     features: ['de', 'en'],
     num_rows: 7
 }),
 'test': Dataset({
     features: ['de', 'en'],
     num_rows: 7
 })}

In [ ]:
# For easier accessibility
train_data, validation_data, test_data = (
    dataset["train"],
    dataset["validation"],
    dataset["test"]
)

In [ ]:
# View each individual pair of english and german text
train_data[0]

{'de': 'Entschuldigung, wie heißt das auf Deutsch?',
 'en': 'Excuse me, how do you say that in German?'}

In [ ]:
# Load in the spaCy models containing the tokenizers
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

**Text preprocessing**

For every English + German sentence pair,
- Apply the spaCy models
- Lowercase all characters. While it may be useful to retain capitalization as all German nouns are capitalized, and there may be differences between a capitalized one and one that is not; "Essen" means a meal/food while "essen" is to eat, lowercasing should simplify things. If evaluation of the model is not good, then perhaps removing lowercasing for German text can be considered.
- The string is lemmatized (reducing words to their base form, called a lemma, eg. "running" -> "run"). Lemmatization is not implemented as it may change the meaning of the sentence.
- Build a list of tokens for each sentence beginning with "sos" and ending with "eos". It is a best practice.


A sample example:

"What a nice weather!" -> ["sos", "what", "a", "nice", "weather", "!", "eos"]

In [ ]:
def tokenize_text(example, en_nlp, de_nlp, sos_token, eos_token):
    en_doc = en_nlp(example["en"])
    de_doc = de_nlp(example["de"])
#   Commented out lemmatization
#    en_tokens = [token.lemma_.lower() for token in en_doc]
#    de_tokens = [token.lemma_ for token in de_doc]
    en_tokens = [token.text for token in en_doc]
    de_tokens = [token.text for token in de_doc]
    en_tokens = [token.lower() for token in en_tokens]
    de_tokens = [token.lower() for token in de_tokens]
    en_tokens = [sos_token] + en_tokens + [eos_token]
    de_tokens = [sos_token] + de_tokens + [eos_token]
    return {"en_tokens": en_tokens, "de_tokens": de_tokens}

In [ ]:
sos_token = "<sos>"
eos_token = "<eos>"

args = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "sos_token": sos_token,
    "eos_token": eos_token,
}

# Map the function to the data to preprocess all sequences
train_data = train_data.map(tokenize_text, fn_kwargs=args)
validation_data = validation_data.map(tokenize_text, fn_kwargs=args)
test_data = test_data.map(tokenize_text, fn_kwargs=args)

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [ ]:
# Check
train_data[0]

{'de': 'Entschuldigung, wie heißt das auf Deutsch?',
 'en': 'Excuse me, how do you say that in German?',
 'en_tokens': ['<sos>',
  'excuse',
  'me',
  ',',
  'how',
  'do',
  'you',
  'say',
  'that',
  'in',
  'german',
  '?',
  '<eos>'],
 'de_tokens': ['<sos>',
  'entschuldigung',
  ',',
  'wie',
  'heißt',
  'das',
  'auf',
  'deutsch',
  '?',
  '<eos>']}

#
### <center> **Constructing the vocabulary** </center>

Key pointers:
- It is important that this vocabulary list is only built on training and not validation/test sets, to avoid information leakage.
- It is to associate every unique token to an integer, as RNNs can only work on numerical values and not string data types.
- The unknown token, denoted by "unk", will be used to replace tokens that are in the validation and test sets but not in the training set (or inside the vocabulary).
- To allow the model to handle unknown tokens by learning to use the context around them, set a `minimum_frequency = 2` parameter, that replaces words appearing fewer than twice in the training set as "unk". In other words, any token that appears less than twice in the training set is replaced by "unk" during the process of converting tokens to indices.
- As models process data in batches, sequences in a batch must be of the same length of tokens. The "pad" token ensures that shorter sentences are padded to match the length of the longest sequence in that batch. Thereafter, the sequences are converted to integers.
- Torchtext's `build_vocab_from_iterator` method can be alternatively used here.

In [ ]:
# Describe the tokens
unk_token = "<unk>"
pad_token = "<pad>"

# Set min_freq to 2
minimum_frequency = 2

# First, add the "special tokens" to the vocab and frequency lists
en_vocab = [unk_token, pad_token, sos_token, eos_token]
de_vocab = [unk_token, pad_token, sos_token, eos_token]
en_freqs = [0, 0, 1, 1]
de_freqs = [0, 0, 1, 1]

# Second, sift through the train data sequences and append unique tokens --> build the vocabulary and respective frequencies.
for i in range(len(train_data)):
    en_text = train_data[i]["en_tokens"]
    for j in range(1, len(en_text)-1):
        if en_text[j] not in en_vocab:
            en_vocab.append(en_text[j])
            en_freqs.append(1)
            continue
        en_freqs[en_vocab.index(en_text[j])] += 1
    de_text = train_data[i]["de_tokens"]
    for k in range(1, len(de_text)-1):
        if de_text[k] not in de_vocab:
            de_vocab.append(de_text[k])
            de_freqs.append(1)
            continue
        de_freqs[de_vocab.index(de_text[k])] += 1

In [ ]:
# Set indexes for the <unk> and <pad> tokens
unk_idx = 0
pad_idx = 1

Define a function that converts the tokens into indices/integers and stores these lists inside each data row.

In [ ]:
# Helper function for training data to denote low frequency (<2) tokens in each text as "<unk>", or 0 in integer
def convert(example, en_vocab, de_vocab):
    en_ids, de_ids = [], []
    en_ids.append(2)
    for i in range(1, len(example["en_tokens"]) - 1):
        if en_freqs[en_vocab.index(example["en_tokens"][i])] < minimum_frequency:
            en_ids.append(0)
        else:
            en_ids.append(en_vocab.index(example["en_tokens"][i]))
    en_ids.append(3)
    de_ids.append(2)
    for j in range(1, len(example["de_tokens"]) - 1):
        if de_freqs[de_vocab.index(example["de_tokens"][j])] < minimum_frequency:
            de_ids.append(0)
        else:
            de_ids.append(de_vocab.index(example["de_tokens"][j]))
    de_ids.append(3)
    return {"en_ids": en_ids, "de_ids": de_ids}


# Separate helper function for validation and test data
def convert_valid(example, en_vocab, de_vocab):
    en_ids, de_ids = [], []
    for token in example["en_tokens"]:
        if token not in en_vocab:
            en_ids.append(0)
        else:
            en_ids.append(en_vocab.index(token))
    for tok in example["de_tokens"]:
        if tok not in de_vocab:
            de_ids.append(0)
        else:
            de_ids.append(de_vocab.index(tok))
    return {"en_ids": en_ids, "de_ids": de_ids}


In [ ]:
# And map the function to all sequences
args = {"en_vocab": en_vocab, "de_vocab": de_vocab}
train_data = train_data.map(convert, fn_kwargs=args)
validation_data = validation_data.map(convert_valid, fn_kwargs=args)
test_data = test_data.map(convert_valid, fn_kwargs=args)

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

In [ ]:
# Check
test_data[0]

{'de': 'Die Distanz ist sehr lang.',
 'en': 'The distance is very long.',
 'en_tokens': ['<sos>', 'the', 'distance', 'is', 'very', 'long', '.', '<eos>'],
 'de_tokens': ['<sos>', 'die', 'distanz', 'ist', 'sehr', 'lang', '.', '<eos>'],
 'en_ids': [2, 45, 0, 16, 85, 47, 34, 3],
 'de_ids': [2, 13, 0, 12, 0, 46, 32, 3]}

Being a `Dataset` object, it is efficient to convert the integers in lists to PyTorch tensors for PyTorch to use them. The `with_format` method used below specifies conversion to PyTorch tensors, the following columns mentioned. To keep all features, set `output_all_columns` to True.

In [ ]:
train_data = train_data.with_format(
    type = "torch",
    columns = ["en_ids", "de_ids"],
    output_all_columns=True
)

validation_data = validation_data.with_format(
    type = "torch",
    columns = ["en_ids", "de_ids"],
    output_all_columns=True,
)

test_data = test_data.with_format(
    type = "torch",
    columns = ["en_ids", "de_ids"],
    output_all_columns=True,
)

In [ ]:
# Check
print(train_data[0])
# should be of torch.Tensor
print(type(train_data[0]["en_ids"]))

{'en_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  3]), 'de_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11,  3]), 'de': 'Entschuldigung, wie heißt das auf Deutsch?', 'en': 'Excuse me, how do you say that in German?', 'en_tokens': ['<sos>', 'excuse', 'me', ',', 'how', 'do', 'you', 'say', 'that', 'in', 'german', '?', '<eos>'], 'de_tokens': ['<sos>', 'entschuldigung', ',', 'wie', 'heißt', 'das', 'auf', 'deutsch', '?', '<eos>']}
<class 'torch.Tensor'>


**Data Loaders**

Preparing a batch of data to be fed to the model:
- Take in a batch of examples and separate the English and German pairs.
- Pass separately into a `pad_sequence` function that takes in a list of tensors and pads each one to the length of the longest tensor. Recall that neural network models require, in a batch, the same length for all sequences. Hence, padding is required.


In [ ]:
def get_collate_fn(pad_idx):
    def collate_fn(batch):
        batch_en_ids = [example["en_ids"] for example in batch]
        batch_de_ids = [example["de_ids"] for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_idx)
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_idx)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [ ]:
def get_data_loader(dataset, batch_size, pad_idx, shuffle=False):
    collate_fn = get_collate_fn(pad_idx)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [ ]:
# How many sequences needed in a batch? Fix this at 5 and 2 respectively as the dataset size is small
train_batch_size = 5
valid_batch_size = 2

# Set shuffle=True for train data
train_data_loader = get_data_loader(train_data, train_batch_size, pad_idx, shuffle=True)
valid_data_loader = get_data_loader(validation_data, valid_batch_size, pad_idx)
test_data_loader = get_data_loader(test_data, valid_batch_size, pad_idx)

#
### <center> **Encoder-Decoder** </center>

Pointers for encoder implementation:
- Notice that no initial hidden/cell state is passed to the RNN. The RNN will automatically create an initial hidden/cell state as a tensor of all zeros.
- Only need the final hidden and cell states for each layer (stacked on top of each other); return these as outputs from the encoder.
- `n_directions` will be 1 here while bidirectional RNNs will have `n_directions = 2`.
- `input_size`: size of the one-hot vectors as input to the encoder (same as the vocabulary size for the input language)
- `embedding_size`: dimensionality of the hidden and cell state


Pointers for decoder implementation:
- Does a single step of decoding; outputs a single token per time-step. The first layer should receive a hidden and cell state from the previous time-step, passing it through the layer with the current embedded token to produce a new hidden and cell state.
- Instead of `input_size`, `output_size`, which is the same size as the vocabulary vector for the target language, is used.
- `nn.Linear` layer that receives the output (the hidden state from the top layer of the RNN) - to predict the next token in the target sequence.
- Input tokens have a sequence length of 1 (decoding one token at a time). Hence, use `unsqueeze(0)`.
- Returns `predictions`, after passing the `outputs` - a hidden state from the top layer of the RNN - through the linear layer, `hidden`, hidden states (one for each layer) and `cell`, cell states (one for each layer).

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, num_layers, d):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = nn.Dropout(d)
        self.embedding = nn.Embedding(input_size, embedding_size)
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout = d)

    def forward(self, x):
        # X shape is as such because we want to send in the full sentence
        # X.shape = (seq_length, batch_size)
        # embedding.shape = (seq_length, batch_size, embedding_size)
        embedding = self.dropout(self.embedding(x))

        # outputs = [seq_length, batch_size, hidden_size * n directions]
        # hidden = [num_layers * n directions, batch_size, hidden_size]
        # cell = [num_layers * n directions, batch_size, hidden_size]
        # outputs are from the top hidden layer for each time-step
        outputs, (hidden, cell) = self.rnn(embedding)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_size, embedding_size, hidden_size,
                 num_layers, d):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = nn.Dropout(d)
        self.output_size = output_size

        self.embedding  = nn.Embedding(output_size, embedding_size)
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=d)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell):
        # Let N be the batch size
        # shape of X = (N), but (1, N) is needed
        # Why not (seq_length, N)? Because prediction of one token at a time
        x = x.unsqueeze(0)

        embedding = self.dropout(self.embedding(x))
        # embedding shape = (1, N, embedding_size)
        outputs, (hidden, cell) = self.rnn(embedding, (hidden, cell))
        # shape of outputs = (1, N, hidden_size)

        predictions = self.fc(outputs)
        # shape of predictions = (1, N, vocab_size)
        predictions = predictions.squeeze(0)

        # Shape of prediction = (N, vocab_size), after getting rid of sentence length dimension
        return predictions, hidden, cell


**Piecing it together**:

Pointers on this implementation:
- It expects an input and target sequence pair (English and German texts respectively).
- It also expects the `teacher_forcing_ratio`, used while training. This means that if the ratio is set to 0.75, the **actual** next token has a 75% probability to be used, with the remaining 25% probability for the **predicted** next token to be used. The chosen token is the next token fed to the decoder in the next time-step.


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, target, teacher_force_ratio):
        batch_size = src.shape[1]
        target_len = target.shape[0]
        target_vocab_size = len(de_vocab)

        # intial tensor to store decoder outputs
        outputs = torch.zeros(target_len, batch_size, target_vocab_size)
        # Get the hidden and cell states from the encoder
        hidden, cell = self.encoder(src)

        # Start token <sos>
        x = target[0]

        # Iterate through, with the number of iterations = len(actual target sequence)
        for i in range(1, target_len):
            output, hidden, cell = self.decoder(x, hidden, cell)

            outputs[i] = output
            # output shape = (N, len(de_vocab))

            # predicted next token in sequence (by doing an argmax over output tensor)
            pred = output.argmax(1)
            # Use actual/predicted as next input depends on teacher forcing
            x = target[i] if random.random() < teacher_force_ratio else pred

        return outputs

#
## <center>**Training Procedure**</center>

Pointers for the training loop:
- Do not include <sos> token during comparison of predicted and actual texts, so apply `[1:]`.
- The loss function works only on 2D inputs with 1D targets, so reshape the output and target tensors.

Pointers for the evaluation loop:
- Switch the model to evaluation mode by specifying `model.eval()`.



In [ ]:
# Training hyperparameters (due to a small dataset size)
num_epochs = 10
lr = 0.001
batch_size = 2

# Model hyperparameters (maybe overkill for a small dataset)
input_size_encoder = len(en_vocab)
input_size_decoder = len(de_vocab)
output_size = len(de_vocab)
embedding_dim = 256
hidden_dim = 512
n_layers = 2
dropout = 0.5

encoder = Encoder(
    input_size_encoder,
    embedding_dim,
    hidden_dim,
    n_layers,
    dropout
)

decoder = Decoder(
    input_size_decoder,
    embedding_dim,
    hidden_dim,
    n_layers,
    dropout
)

model = Seq2Seq(encoder, decoder)

The code block below is optional to be run. It shows the number of parameters in the encoder and decoder blocks, and the total number of parameters.

In [ ]:
# Initialization of all weights from a uniform distribution in (-0.08,0.08)
def init_weights(mod):
    for name, param in mod.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

def count_parameters(mod):
    return sum(p.numel() for p in mod.parameters() if p.requires_grad)


model.apply(init_weights)
# print(count_parameters(model))

In [ ]:
# Adam optimizer to update parameters in the training loop
optimizer = optim.Adam(model.parameters())

# Loss to calculate the average loss per token, using cross-entropy loss
# Ignore the loss when the target token is a padding one
loss_fn = nn.CrossEntropyLoss(ignore_index = pad_idx)

In [ ]:
# Training loop
def train_fn(
    model, data_loader, optimizer, criterion, teacher_forcing_ratio
):
    model.train()
    epoch_loss = 0
    for i, batch in enumerate(data_loader):
        # Reinitialize the optimizer gradients
        optimizer.zero_grad()

        # Get the input and target sentences of a batch
        input = batch["en_ids"]
        target = batch["de_ids"]

        # Get the model prediction
        pred = model(input, target, teacher_forcing_ratio)

        # Not including the "sos" token at the start, for both prediction and target
        pred = pred[1:].reshape(-1, pred.shape[-1])
        target = target[1:].reshape(-1)
        # Compute the cross entropy loss over all tokens. This is not averaged.
        loss = criterion(pred, target)

        # Calculate gradients by backpropagation
        loss.backward()

        # To avoid exploding gradients, clip by norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        # Update model parameters
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

In [ ]:
# Evaluation loop
def evaluate_fn(model, data_loader, criterion):
    model.eval()
    epoch_loss = 0

    # Ensure no gradients are calculated within the block
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            input = batch["en_ids"]
            target = batch["de_ids"]
            pred = model(input, target, 0)
            pred = pred[1:].reshape(-1, pred.shape[-1])
            target = target[1:].reshape(-1)

            loss = criterion(pred, target)
            epoch_loss += loss.item()
    return epoch_loss / len(data_loader)

Below, train the model with the goal of translating an English sentence into German.

In [ ]:
# Real training loop
# loss counter (check if the model achieves the best validation loss so far)
min_loss = 999
teacher_forcing_ratio = 0.5

for epoch in range(num_epochs):
    print(f"Epoch [{epoch}/{num_epochs}]")

    train_loss = train_fn(
    model, train_data_loader, optimizer, loss_fn, teacher_forcing_ratio
)
    val_loss = evaluate_fn(model, valid_data_loader, loss_fn)

    if val_loss < min_loss:
        min_loss = val_loss
        # keep check of the best model parameters
        dt = model.state_dict()
    print(f"\tTrain Loss: {train_loss:7.3f}")
    print(f"\tValidation Loss: {val_loss:7.3f}")

Epoch [0/10]
	Train Loss:   4.153
	Validation Loss:   4.180
Epoch [1/10]
	Train Loss:   3.454
	Validation Loss:   4.037
Epoch [2/10]
	Train Loss:   3.243
	Validation Loss:   3.896
Epoch [3/10]
	Train Loss:   3.184
	Validation Loss:   3.960
Epoch [4/10]
	Train Loss:   3.088
	Validation Loss:   4.008
Epoch [5/10]
	Train Loss:   2.960
	Validation Loss:   4.108
Epoch [6/10]
	Train Loss:   2.824
	Validation Loss:   4.047
Epoch [7/10]
	Train Loss:   2.753
	Validation Loss:   4.064
Epoch [8/10]
	Train Loss:   2.655
	Validation Loss:   3.823
Epoch [9/10]
	Train Loss:   2.634
	Validation Loss:   3.886


In [ ]:
# Save the best performing model at the end of training
torch.save(dt, "new_mod.pt")

#
## <center>**Model Testing**</center>

Having successfully saved the model parameters that give the lowest validation loss in `best_mod.pt`, load these parameters and run this model on the test set.

Note: Read about saving and loading models in PyTorch in https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html, which provides context into `torch.load` and more

In [ ]:
# Test on the test data set
model.load_state_dict(torch.load("new_mod.pt"))
test_loss = evaluate_fn(model, test_data_loader, loss_fn)
print(f"Test Loss: {test_loss:.3f} ")

# Compare with the validation data set face to face
v_loss = evaluate_fn(model, valid_data_loader, loss_fn)
print(f"Validation Loss: {v_loss:.3f} ")

Test Loss: 3.742 
Validation Loss: 3.886 


If the test loss is similar to the validation loss, it is a good indicator of little to no overfitting on the validation set. With a difference of ~0.1 between both values, it appears there is little overfitting on the validation set.

Suppose we want to get the model to predict a personal text that we feed into the model. To do so,

In [ ]:
# Set an upper limit for the output length (for the model to reach a limit in translating sequence)
max_output_length = 15

def process_message(text, en_vocab, de_vocab):
    # Load in the model
    model.load_state_dict(torch.load("new_mod.pt"))
    model.eval()
    with torch.no_grad():
        # Preprocess the sentence into a list of tokens
        en_msg = en_nlp(text)
        en_tokens = [token.text for token in en_msg]
        en_tokens = [token.lower() for token in en_tokens]
        en_tokens = [sos_token] + en_tokens + [eos_token]

        # Convert the tokens into indices
        en_ids = []
        for token in en_tokens:
            if token not in en_vocab:
                en_ids.append(0)
            else:
                en_ids.append(en_vocab.index(token))
        tensor = torch.LongTensor(en_ids).unsqueeze(-1)

        # Pass tensor through encoder to get hidden + cell states
        hidden, cell = model.encoder(tensor)

        # Pass in the first "sos" token as a tensor into decoder
        inputs = [2]
        for _ in range(max_output_length):
            inputs_tensor = torch.LongTensor([inputs[-1]])
            output, hidden, cell = model.decoder(inputs_tensor, hidden, cell)

            # Get predicted token the model presumes that is most likely next in the sequence
            predicted_token = output.argmax(-1).item()
            inputs.append(predicted_token)
            # If predicted token from decoder is "eos", exit. Otherwise, continue translating until hitting the max output length
            if predicted_token == de_vocab.index(eos_token):
                break

        # Convert the indices into tokens using the vocabulary
        translated = ""
        for i in range(1, len(inputs)-1):
            translated += de_vocab[inputs[i]]
    return translated

In [ ]:
# Scenario 1: Mostly unknown words (words not appeared in the dataset)
text = "The office is on the second floor, to the right."
print(process_message(text, en_vocab, de_vocab))

guten,<unk><unk><unk><unk>?


In [ ]:
# Scenario 2:  All known words that appear in the dataset
text = "Good day, what is your number?"
print(process_message(text, en_vocab, de_vocab))

guten,,<unk><unk><unk>?


#
## **<center>Model Evaluation</center>**

From the above test scenarios and the test loss, the model has a pretty bad performance. To evaluate the model in another way, the BLEU score is a metric used to measure how similar the machine-generated translation is compared to human reference translations.

The BLEU metric can also be used to compare different translation models and versions.

In [ ]:
# Load the BLEU metric
import evaluate
bleu = evaluate.load("bleu")

Compare the predictions with the actual results. Take Scenario 2 predictions and output as the test case.

Note: `predictions` should be a list of strings and `references` a list of list-of-strings.

In [ ]:
def tokenize(text):
    return text.split()

In [ ]:
# Pass the predictions, references and function into the BLEU metric's compute method to get the score
prediction = ["guten,,<unk><unk><unk>?"]
actual = ["guten tag, wie ist deine handynummer?"]
results = bleu.compute(
    predictions=prediction, references=actual, tokenizer=tokenize
)

In [ ]:
# Return the results (BLEU score is the first value)
results

{'bleu': 0.0,
 'precisions': [0.0, 0.0, 0.0, 0.0],
 'brevity_penalty': 0.006737946999085467,
 'length_ratio': 0.16666666666666666,
 'translation_length': 1,
 'reference_length': 6}

BLEU takes on a value between 0 and 1, and a higher value is better, which means the model has a high correlation with human judgement, making it a good translation model.

For this model, the BLEU score is 0.

#
## <center>**Reflection**</center>

As seen in this notebook, a limited dataset size does not give the model any flexibility of building a proper sentence; there is a higher frequency of words replaced with "unknown tokens". As the model learns relationships (word after word) between unknown tokens and normal words (or even unknown token after unknown token), it has a strong tendency to build a sentence comprising only unknown tokens.

When the prediction containing almost all unknown tokens is compared to the actual translated text, it yields a very low BLEU score. Even without the BLEU score, testing the model with different input sentences proves that it is very poor at translating.

In [ ]:
# As seen in the frequency of english and german words in the vocabulary, there are many words (frequency of 1) replaced by unknown tokens
print(en_freqs)
print(de_freqs)

[0, 0, 1, 1, 3, 3, 27, 7, 4, 15, 2, 7, 3, 3, 30, 4, 13, 3, 1, 4, 5, 4, 2, 10, 1, 2, 2, 4, 1, 8, 6, 14, 4, 7, 21, 8, 5, 6, 2, 4, 4, 2, 2, 1, 2, 13, 1, 2, 2, 1, 1, 2, 1, 4, 4, 7, 1, 1, 1, 4, 1, 2, 2, 3, 3, 1, 1, 2, 5, 2, 1, 4, 2, 1, 1, 3, 3, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 1, 1, 3, 27, 11, 2, 8, 3, 3, 30, 13, 6, 3, 1, 3, 4, 2, 7, 1, 2, 2, 4, 2, 3, 1, 6, 4, 4, 2, 1, 21, 8, 4, 6, 2, 4, 3, 4, 1, 5, 2, 1, 1, 1, 2, 2, 1, 1, 12, 2, 1, 4, 5, 2, 1, 1, 1, 6, 4, 1, 2, 2, 3, 2, 2, 1, 2, 3, 2, 1, 2, 2, 1, 1, 1, 3, 2, 2, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


#
### <center>**Possible Improvements**</center>

- The biggest problem is the dataset size. A deep-learning model should be trained on a much larger dataset.
- The model's architecture could be improved: introducing attention
- Training procedure could be tweaked: number of epochs, batch size, optimizer, loss function and others.